In [1]:
import os
import json
import hashlib
import pandas as pd

In [7]:
# Load the JSON dataset from the data folder
with open("../data/raw_credit_applications.json", "r") as f:
    data = json.load(f)

# Flatten nested JSON structure
df = pd.json_normalize(data)

print("Dataset loaded")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Dataset loaded
Rows: 502
Columns: ['_id', 'spending_behavior', 'processing_timestamp', 'applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn', 'applicant_info.ip_address', 'applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code', 'financials.annual_income', 'financials.credit_history_months', 'financials.debt_to_income', 'financials.savings_balance', 'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate', 'decision.approved_amount', 'financials.annual_salary', 'notes']


In [8]:
# Secret pepper used to strengthen the hash
PEPPER = "ORION_GOV_PEPPER_2026!x7"

# Pseudonymization function
def pseudo(x):
    if pd.isna(x) or x == "":
        return x
    value = (str(x).strip() + PEPPER).encode("utf-8")
    return hashlib.sha256(value).hexdigest()

# Columns to pseudonymize
cols_to_hash = [
    "applicant_info.ssn",
    "applicant_info.email",
    "applicant_info.full_name"
]

# Apply pseudonymization
for c in cols_to_hash:
    if c in df.columns:
        df[c + "_hashed"] = df[c].apply(pseudo)

# Remove original PII columns (data minimization)
df = df.drop(columns=[c for c in cols_to_hash if c in df.columns])

print("Pseudonymization completed")
print(df.head())

Pseudonymization completed
       _id                                  spending_behavior  \
0  app_200  [{'category': 'Shopping', 'amount': 480}, {'ca...   
1  app_037  [{'category': 'Rent', 'amount': 608}, {'catego...   
2  app_215              [{'category': 'Rent', 'amount': 109}]   
3  app_024           [{'category': 'Fitness', 'amount': 575}]   
4  app_184     [{'category': 'Entertainment', 'amount': 463}]   

   processing_timestamp applicant_info.ip_address applicant_info.gender  \
0  2024-01-15T00:00:00Z            192.168.48.155                  Male   
1                   NaN              10.1.102.112                     M   
2                   NaN            10.240.193.250                  Male   
3                   NaN            192.168.175.67                  Male   
4  2024-01-15T00:00:00Z            172.29.125.105                     M   

  applicant_info.date_of_birth applicant_info.zip_code  \
0                   2001-03-09                   10036   
1              